In [ ]:
!pip install transformers

In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 19187.72it/s]


In [1]:
import torch

# =========================
# 1. VOCABULARY
# =========================

print("=" * 60)
print("VOCABULARY")
print("=" * 60)

print("Vocabulary size:", tokenizer.vocab_size)

vocab = tokenizer.get_vocab()
print(vocab.items())

print("\nSample vocabulary:")
for token, token_id in list(vocab.items())[0:10]:  # Display first 10 tokens
    print(f"{token!r:20} -> {token_id}")






VOCABULARY


NameError: name 'tokenizer' is not defined

In [26]:

# =========================
# 4. SAMPLE ACTUAL WEIGHTS
# =========================

print("\n" + "=" * 60)
print("SAMPLE WEIGHT VALUES")
print("=" * 60)


for name, param in list(model.named_parameters())[:10]:

    print(f"\n{name}")
    print("Shape:", tuple(param.shape))

    tensor = param.detach().cpu()

    if tensor.ndim == 1:
        # 1D parameter
        print("Sample values:")
        print(tensor[:10])

    elif tensor.ndim == 2:
        # 2D parameter
        print("Sample values (5 x 5):")
        print(tensor[:5, :5])

    elif tensor.ndim == 3:
        # 3D parameter
        print("Sample values (first 2 x 3 x 3):")
        print(tensor[:2, :3, :3])

    else:
        print("Tensor dimensions:", tensor.ndim)
        print("First 10 values:")
        print(tensor.flatten()[:10])


SAMPLE WEIGHT VALUES

model.embed_tokens.weight
Shape: (151936, 896)
Sample values (5 x 5):
tensor([[-1.0376e-02,  4.0771e-02,  9.7046e-03,  6.9618e-05, -2.7100e-02],
        [-1.4587e-02, -1.3657e-03, -1.7700e-02, -2.6703e-03,  3.7079e-03],
        [-3.6621e-02, -1.0193e-02,  7.8125e-03, -1.0925e-02,  8.0566e-03],
        [-9.3384e-03, -1.2085e-02, -1.5381e-02,  1.0864e-02,  3.9673e-03],
        [-9.5215e-03,  4.2114e-03,  6.0120e-03, -1.8433e-02,  6.4087e-03]],
       dtype=torch.bfloat16)

model.layers.0.self_attn.q_proj.weight
Shape: (896, 896)
Sample values (5 x 5):
tensor([[-0.0019, -0.0052,  0.0188,  0.0125,  0.0040],
        [ 0.0084,  0.0018,  0.0435,  0.0183,  0.0003],
        [-0.0168, -0.0248,  0.0422,  0.0344, -0.0064],
        [-0.0101,  0.0113, -0.0310,  0.0042,  0.0439],
        [ 0.0479, -0.0703, -0.0010,  0.0771,  0.0562]], dtype=torch.bfloat16)

model.layers.0.self_attn.q_proj.bias
Shape: (896,)
Sample values:
tensor([-1.4954e-02,  2.5513e-02, -1.0352e-01, -1.3574e-

In [32]:
import torch

prompt = "The captital of france is the"

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt")

print("Input tokens:")
for token_id in inputs["input_ids"][0]:
    print(token_id.item(), repr(tokenizer.decode([token_id.item()])))

# Run through GPT-2
with torch.no_grad():
    outputs = model(**inputs)

# Get logits for the LAST input token
logits = outputs.logits[0, -1, :]

# Convert logits -> probabilities
probabilities = torch.softmax(logits, dim=-1)

# Find highest probability
probability, token_id = torch.max(probabilities, dim=0)

print("\n" + "=" * 50)
print("HIGHEST PROBABILITY TOKEN")
print("=" * 50)

print("Token ID:", token_id.item())
print("Token:", repr(tokenizer.decode([token_id.item()])))
print("Probability:", probability.item())
print("Probability (%):", probability.item() * 100)

Input tokens:
785 'The'
6427 ' capt'
2174 'ital'
315 ' of'
47587 ' france'
374 ' is'
279 ' the'

HIGHEST PROBABILITY TOKEN
Token ID: 6722
Token: ' capital'
Probability: 0.19921875
Probability (%): 19.921875
